But what counts as “too small”? Don’t guess. Measure it. Load your JSON and look at the actual block lengths


In [1]:
import json
with open("../data/processed/cleaned_pages.json") as f:
    pages = json.load(f)

lengths = []
for game, game_pages in pages.items():
    for page_number, text in game_pages:
        for block in text.split("\n\n"):
            lengths.append(len(block))

lengths.sort()
print(len(lengths), "blocks")
print("shortest 20:", lengths[:20])
print("median:", lengths[len(lengths)//2])

728 blocks
shortest 20: [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
median: 26


In [2]:
tiny = []
for game, game_pages in pages.items():
    for page_number, text in game_pages:
        for block in text.split("\n\n"):
            if len(block) <= 3:
                tiny.append((game, page_number, repr(block)))

for t in tiny[:30]:
    print(t)
print(len(tiny), "blocks of 3 chars or fewer")

('catan', 2, "'2'")
('catan', 3, "'3'")
('catan', 3, "'3'")
('catan', 3, "'6'")
('catan', 3, "'1'")
('catan', 3, "'2'")
('catan', 3, "'5'")
('catan', 3, "'4'")
('catan', 3, "'2:1'")
('catan', 3, "'2:1'")
('catan', 3, "'2:1'")
('catan', 3, "'2:1'")
('catan', 3, "'3:1'")
('catan', 3, "'3:1'")
('catan', 3, "'2:1'")
('catan', 3, "'3:1'")
('catan', 3, "'3:1'")
('catan', 3, "'2:1'")
('catan', 3, "'2:1'")
('catan', 3, "'2'")
('catan', 3, "'5'")
('catan', 3, "'6'")
('catan', 3, "'1'")
('catan', 3, "'4'")
('catan', 3, "'3'")
('catan', 3, "'2:1'")
('catan', 3, "'3:1'")
('catan', 3, "'3:1'")
('catan', 3, "'3:1'")
('catan', 3, "'3:1'")
73 blocks of 3 chars or fewer


In [5]:
n = len(lengths)
print("lenghts:",n)
print("75th percentile:", lengths[int(n * 0.75)])
print("90th percentile:", lengths[int(n * 0.90)])
print("longest:", lengths[-1])

lenghts: 728
75th percentile: 121
90th percentile: 262
longest: 2433


In [7]:
def chunk_page(text, game, page_number, target=800):
    chunks = []
    current = []          # blocks in the chunk being built
    current_len = 0

    for block in text.split("\n\n"):
        if not block.strip():                 # skip empty blocks
            continue

        # adding this block would overflow, so save the current chunk first
        if current_len + len(block) + 2 > target and current:
            chunks.append({
                "id": f"{game}-p{page_number}-c{len(chunks)}",
                "game": game,
                "page": page_number,
                "text": "\n\n".join(current),
            })
            current = []
            current_len = 0

        current.append(block)
        current_len += len(block) + 2

    # after the loop: save whatever is left (the end of the page)
    if current:
        chunks.append({
            "id": f"{game}-p{page_number}-c{len(chunks)}",
            "game": game,
            "page": page_number,
            "text": "\n\n".join(current),
        })

    return chunks

In [8]:
text = pages["blackjack"][0][1]
chunks = chunk_page(text, "blackjack", 1)
for c in chunks:
    print(c["id"], len(c["text"]))
    print(c["text"][:150], "\n---")

# every non-empty block should appear in exactly one chunk
blocks = [b for b in text.split("\n\n") if b.strip()]
chunked = sum(len([b for b in c["text"].split("\n\n") if b.strip()]) for c in chunks)
print("blocks:", len(blocks), "| in chunks:", chunked)

blackjack-p1-c0 793
Blackjack is a fast, social casino classic where players compete only against the dealer, not each other. The goal is simple: build a hand closer to 2 
---
blackjack-p1-c1 348
Players Turn

EVALUATE YOUR HAND

Add your two card values together

CHOOSE YOUR ACTION

● Hit: Take another card to add to your total ● Stand/Stay: K 
---
blocks: 17 | in chunks: 17


In [11]:
all_chunks = []
for game, game_pages in pages.items():
    for page_number, text in game_pages:
        all_chunks.extend(chunk_page(text, game, page_number))


print(len(all_chunks), "chunks total")

counts = {}
for c in all_chunks:
    game = c["game"]
    counts[game] = counts.get(game, 0) + 1
print(counts)

113 chunks total
{'catan': 32, 'monopoly': 14, 'blackjack': 4, 'poker': 30, 'rummy': 21, 'texas_holdem': 12}


In [12]:
lengths = [len(c["text"]) for c in all_chunks]
print("shortest:", min(lengths), "| longest:", max(lengths))

shortest: 31 | longest: 2433


In [13]:
  longest = max(all_chunks, key=lambda c: len(c["text"]))
  print(longest["id"])
  print(longest["text"][:400])

rummy-p7-c0
longer to reach the target winning score. When playing this way it might be preferable to set a limit on the number of rounds instead of using a target score. 6. Lowest score wins: This is similar to the above variant except points are added instead of deducted. At the end of a hand players add up their deadwood score and it is added to their own total. The game ends when any player reaches a pred


In [14]:
def find_chunks(keyword):
    for c in all_chunks:
        if keyword.lower() in c["text"].lower():
            print(c["id"], "|", c["text"][:200].replace("\n", " "), "\n---")

In [17]:
find_chunks("Free Parking")

find_chunks("hole cards")
find_chunks("BUILDING COST")

monopoly-p1-c0 | BRAND  1 2  TOP TIPS To keep your game short and sweet, don't use the house rules!  If you're still in debt, you are bankrupt and out of the game!  BUILDINGS DEALS & TRADES  Always auction when someon 
---
monopoly-p2-c5 | •	 You can double the rent for those streets! •	 You may build Houses and Hotels and charge even more rent! See BUILDINGS. Owned Properties When you land on a property that somebody else owns, the own 
---
texas_holdem-p2-c1 | In Hold'em, each player is dealt two private cards (known as ‘hole cards') that belong to them alone. Five community cards are dealt face-up, to form the ‘board'. All players in the game use these sha 
---
texas_holdem-p2-c3 | In Pot Limit and No Limit games, the games are referred to by the size of their blinds (for example, a 2 Hold'em game has a small blind of $1 and a big blind of $2). 1/  Depending on the exact structu 
---
texas_holdem-p3-c1 | After seeing his or her hole cards, each player now has the option to play his 